In [18]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.home() / "Github" / "the-bible-catalog"))

In [19]:
from config.settings import *
from config._common_libraries import *
from config._common_functions import *
from config._util_functions import *

In [29]:
import os
import re
import time
import requests
import pandas as pd

# ─────────────────────────────────────────────
# Bible Canon — 66 Books
# ─────────────────────────────────────────────

BIBLE_CANON = [
    # ── Old Testament (39) ──────────────────
    ("Genesis",         "Old Testament", 50),
    ("Exodus",          "Old Testament", 40),
    ("Leviticus",       "Old Testament", 27),
    ("Numbers",         "Old Testament", 36),
    ("Deuteronomy",     "Old Testament", 34),
    ("Joshua",          "Old Testament", 24),
    ("Judges",          "Old Testament", 21),
    ("Ruth",            "Old Testament",  4),
    ("1 Samuel",        "Old Testament", 31),
    ("2 Samuel",        "Old Testament", 24),
    ("1 Kings",         "Old Testament", 22),
    ("2 Kings",         "Old Testament", 25),
    ("1 Chronicles",    "Old Testament", 29),
    ("2 Chronicles",    "Old Testament", 36),
    ("Ezra",            "Old Testament", 10),
    ("Nehemiah",        "Old Testament", 13),
    ("Esther",          "Old Testament", 10),
    ("Job",             "Old Testament", 42),
    ("Psalms",          "Old Testament",150),
    ("Proverbs",        "Old Testament", 31),
    ("Ecclesiastes",    "Old Testament", 12),
    ("Song of Solomon", "Old Testament",  8),
    ("Isaiah",          "Old Testament", 66),
    ("Jeremiah",        "Old Testament", 52),
    ("Lamentations",    "Old Testament",  5),
    ("Ezekiel",         "Old Testament", 48),
    ("Daniel",          "Old Testament", 12),
    ("Hosea",           "Old Testament", 14),
    ("Joel",            "Old Testament",  3),
    ("Amos",            "Old Testament",  9),
    ("Obadiah",         "Old Testament",  1),
    ("Jonah",           "Old Testament",  4),
    ("Micah",           "Old Testament",  7),
    ("Nahum",           "Old Testament",  3),
    ("Habakkuk",        "Old Testament",  3),
    ("Zephaniah",       "Old Testament",  3),
    ("Haggai",          "Old Testament",  2),
    ("Zechariah",       "Old Testament", 14),
    ("Malachi",         "Old Testament",  4),
    # ── New Testament (27) ──────────────────
    ("Matthew",         "New Testament", 28),
    ("Mark",            "New Testament", 16),
    ("Luke",            "New Testament", 24),
    ("John",            "New Testament", 21),
    ("Acts",            "New Testament", 28),
    ("Romans",          "New Testament", 16),
    ("1 Corinthians",   "New Testament", 16),
    ("2 Corinthians",   "New Testament", 13),
    ("Galatians",       "New Testament",  6),
    ("Ephesians",       "New Testament",  6),
    ("Philippians",     "New Testament",  4),
    ("Colossians",      "New Testament",  4),
    ("1 Thessalonians", "New Testament",  5),
    ("2 Thessalonians", "New Testament",  3),
    ("1 Timothy",       "New Testament",  6),
    ("2 Timothy",       "New Testament",  4),
    ("Titus",           "New Testament",  3),
    ("Philemon",        "New Testament",  1),
    ("Hebrews",         "New Testament", 13),
    ("James",           "New Testament",  5),
    ("1 Peter",         "New Testament",  5),
    ("2 Peter",         "New Testament",  3),
    ("1 John",          "New Testament",  5),
    ("2 John",          "New Testament",  1),
    ("3 John",          "New Testament",  1),
    ("Jude",            "New Testament",  1),
    ("Revelation",      "New Testament", 22),
]

# Lookup: book_name → (testament, chapter_count)
BOOK_LOOKUP = {book: (testament, chapters) for book, testament, chapters in BIBLE_CANON}

# ─────────────────────────────────────────────
# ESV API Parameters
# Tuned for clean AI/NLP processing
# ─────────────────────────────────────────────

ESV_DEFAULT_PARAMS = {
    "include-passage-references":  False,  # Tracked in schema
    "include-verse-numbers":       True,   # Required for verse parsing
    "include-first-verse-numbers": True,
    "include-footnotes":           False,  # Clean text for AI
    "include-footnote-body":       False,
    "include-headings":            False,  # Clean text for AI
    "include-short-copyright":     False,  # Added at display layer
    "include-selahs":              True,   # Preserve Psalm poetic markers
    "indent-using":                "space",
    "line-length":                 0,      # No artificial line wrapping
}


# ─────────────────────────────────────────────
# ESVBibleIngestion Class
# ─────────────────────────────────────────────

class ESVBibleIngestion:
    """
    ESV Bible ingestion client for TrioTheo Bronze layer.

    Schema:
        translation  (str)  : "ESV"
        testament    (str)  : "Old Testament" | "New Testament"
        book         (str)  : Full book name
        chapter      (int)  : Chapter number
        verse_number (int)  : Verse number
        verse_text   (str)  : Clean verse text
    """

    BASE_URL      = "https://api.esv.org/v3/passage/text/"
    RATE_LIMIT    = 0.25   # seconds between requests
    MAX_RETRIES   = 3
    RETRY_BACKOFF = 2.0

    def __init__(self, api_key: str = None, rate_limit: float = None):
        self.api_key    = api_key or os.environ.get("ESV_API_KEY", "")
        if not self.api_key:
            raise EnvironmentError(
                "ESV API key required. Pass api_key= or set ESV_API_KEY env var."
            )
        self.rate_limit = rate_limit or self.RATE_LIMIT
        self._headers   = {"Authorization": f"Token {self.api_key}"}

    # ── Public ───────────────────────────────────────────────────

    def get_passage_df(self, passage: str) -> pd.DataFrame:
        """
        Fetch any passage reference and return a verse-per-row DataFrame.
        e.g. "Genesis 1-3", "John 3:16", "Romans 8:1-4"
        """
        raw      = self._fetch_raw(passage)
        passages = raw.get("passages", [])
        if not passages:
            print(f"  ⚠ No passage data returned for: {passage!r}")
            return self._empty_df()

        canonical       = raw.get("canonical", passage)
        book, testament = self._resolve_book_testament(canonical, passage)
        rows            = self._parse_passage_text(passages[0], book, testament)
        return self._to_df(rows)

    def get_book_df(self, book_name: str) -> pd.DataFrame:
        """
        Fetch all chapters of a single book.
        e.g. "Romans", "1 Corinthians", "Philemon"
        """
        if book_name not in BOOK_LOOKUP:
            raise ValueError(f"Unknown book: {book_name!r}. Check BIBLE_CANON for valid names.")

        testament, chapter_count = BOOK_LOOKUP[book_name]
        all_rows = []

        for chapter_num in range(1, chapter_count + 1):
            # Use verse range format to guarantee full chapter is returned.
            # Plain "Book N" can be interpreted as chapter N verse 1 by the API.
            ref      = f"{book_name} {chapter_num}:1-200"
            raw      = self._fetch_raw(ref)
            passages = raw.get("passages", [])

            if passages:
                all_rows.extend(self._parse_passage_text(passages[0], book_name, testament))
            else:
                print(f"  ⚠ No data for {ref}")

            time.sleep(self.rate_limit)

        return self._to_df(all_rows)

    def get_full_bible_df(self, progress: bool = True) -> pd.DataFrame:
        """
        Ingest all 66 books of the ESV Bible.
        ~1,189 API calls, ~7 min at default rate limit.
        """
        all_rows = []

        for book_idx, (book_name, testament, chapter_count) in enumerate(BIBLE_CANON, 1):
            book_rows = []

            if progress:
                print(f"[{book_idx:02d}/66] {book_name}...", end=" ", flush=True)

            for chapter_num in range(1, chapter_count + 1):
                # Use verse range format to guarantee full chapter is returned.
                ref      = f"{book_name} {chapter_num}:1-200"
                raw      = self._fetch_raw(ref)
                passages = raw.get("passages", [])

                if passages:
                    book_rows.extend(self._parse_passage_text(passages[0], book_name, testament))
                else:
                    print(f"  ⚠ Empty response: {ref}")

                time.sleep(self.rate_limit)

            all_rows.extend(book_rows)

            if progress:
                print(f"✓ {len(book_rows)} verses")

        return self._to_df(all_rows)

    # ── Private: API ─────────────────────────────────────────────

    def _fetch_raw(self, passage: str) -> dict:
        params = {"q": passage, **ESV_DEFAULT_PARAMS}

        for attempt in range(1, self.MAX_RETRIES + 1):
            try:
                resp = requests.get(
                    self.BASE_URL, headers=self._headers,
                    params=params, timeout=15
                )
                resp.raise_for_status()
                return resp.json()

            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 400:
                    print(f"  ⚠ Bad request — skipping: {passage!r}")
                    return {}

            except requests.exceptions.RequestException as e:
                print(f"  ⚠ Request error on {passage!r}, attempt {attempt}: {e}")

            if attempt < self.MAX_RETRIES:
                time.sleep(self.rate_limit * (self.RETRY_BACKOFF ** attempt))

        print(f"  ✗ All {self.MAX_RETRIES} attempts failed for {passage!r}")
        return {}

    # ── Private: Parsing ─────────────────────────────────────────

    def _parse_passage_text(self, passage_text: str, book: str, testament: str) -> list[dict]:
        """
        Split ESV API passage text into one dict per verse.
        Handles prose, poetry (Psalms/Proverbs), Selah markers, and multi-chapter passages.
        """
        if not passage_text:
            return []

        parts = re.split(r'\[(\d+)\]', passage_text)
        rows  = []
        i     = 1

        while i < len(parts) - 1:
            verse_num_str = parts[i].strip()
            raw_text      = parts[i + 1]

            verse_text = re.sub(r'[ \t]+', ' ', raw_text)
            verse_text = re.sub(r'\n+', ' ', verse_text)
            verse_text = re.sub(r'\s{2,}[A-Z][A-Z\s,]+$', '', verse_text).strip()

            if verse_num_str.isdigit() and verse_text:
                rows.append({
                    "translation":  "ESV",
                    "testament":    testament,
                    "book":         book,
                    "chapter":      None,
                    "verse_number": int(verse_num_str),
                    "verse_text":   verse_text,
                })
            i += 2

        # Resolve chapter numbers by detecting verse_number resets to 1
        if rows:
            chapter = 1
            for idx, row in enumerate(rows):
                if idx > 0 and row["verse_number"] == 1:
                    chapter += 1
                row["chapter"] = chapter

        return rows

    def _resolve_book_testament(self, canonical: str, fallback: str) -> tuple[str, str]:
        for book_name in BOOK_LOOKUP:
            if canonical.startswith(book_name):
                testament, _ = BOOK_LOOKUP[book_name]
                return book_name, testament
        for book_name in BOOK_LOOKUP:
            if fallback.startswith(book_name):
                testament, _ = BOOK_LOOKUP[book_name]
                return book_name, testament
        return "Unknown", "Unknown"

    # ── Private: DataFrame ───────────────────────────────────────

    def _to_df(self, rows: list[dict]) -> pd.DataFrame:
        if not rows:
            return self._empty_df()

        df = pd.DataFrame(rows, columns=[
            "translation", "testament", "book",
            "chapter", "verse_number", "verse_text",
        ])
        df["chapter"]      = df["chapter"].astype("Int16")
        df["verse_number"] = df["verse_number"].astype("Int16")
        df["translation"]  = df["translation"].astype("category")
        df["testament"]    = df["testament"].astype("category")
        df["book"]         = df["book"].astype("string")
        df["verse_text"]   = df["verse_text"].astype("string")

        return df.reset_index(drop=True)

    def _empty_df(self) -> pd.DataFrame:
        return pd.DataFrame(columns=[
            "translation", "testament", "book",
            "chapter", "verse_number", "verse_text"
        ])

In [30]:
ingestor = ESVBibleIngestion(api_key=EXTERNAL_SERVICES["esv_api_token"])

In [ ]:
df = ingestor.get_full_bible_df() # TODO: run to test/save to csv so I do not need to redo this

In [ ]:
display(df)